In [1]:
!pip uninstall -y transformers sentence-transformers pillow pdfplumber

Found existing installation: transformers 4.55.4
Uninstalling transformers-4.55.4:
ERROR: Exception:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/shutil.py", line 847, in move
    os.rename(src, real_dst)
FileNotFoundError: [Errno 2] No such file or directory: '/Users/dipeshkardate/Desktop/AI-Powered-Resume-Screening-And-Job-Matching-System/venv/bin/transformers' -> '/private/var/folders/d5/_xxsp4_d0716jz7wj0717pkr0000gn/T/pip-uninstall-as96rxys/transformers'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/dipeshkardate/Desktop/AI-Powered-Resume-Screening-And-Job-Matching-System/venv/lib/python3.12/site-packages/pip/_internal/cli/base_command.py", line 110, in _run_wrapper
    status = _inner_run()
             ^^^^^^^^^^^^
  File "/Users/dipeshkardate/Desktop/AI-Powered-Resume-Screening-And-Job-Matching-System/venv/li

In [2]:
!pip install --no-cache-dir \
transformers==4.55.4 \
sentence-transformers==5.1.0 \
pillow==12.3.0 \
pdfplumber==0.11.10 \
python-docx \
faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 537.0 kB/s  0:00:20m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 929.7 kB/s  0:00:05 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sentence-transformers]ence-transformers]


In [3]:
import sys
print(sys.version)

import PIL
print(PIL.__version__)

import transformers
print(transformers.__version__)

import sentence_transformers
print(sentence_transformers.__version__)

3.12.13 (main, Mar  3 2026, 12:39:30) [Clang 21.0.0 (clang-2100.0.123.102)]
12.3.0


/Users/dipeshkardate/Desktop/AI-Powered-Resume-Screening-And-Job-Matching-System/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.55.4
5.1.0


In [4]:
import os
import re
import numpy as np
import pandas as pd
import torch
import faiss
import pdfplumber
import docx

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

import spacy
from spacy.matcher import PhraseMatcher

In [5]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
# when not using colab
# BASE_PATH = "/content/drive/MyDrive/AI Resume Screening/"

# DATA_PATH = BASE_PATH
# MODEL_PATH = BASE_PATH + "models/"
# OUTPUT_PATH = BASE_PATH + "outputs/"


# job_df = pd.read_csv(DATA_PATH + "processed_jobs.csv")

# index = faiss.read_index(MODEL_PATH + "job_faiss.index")

# print("Total Jobs:", len(job_df))
# print("FAISS Index:", index.ntotal)

In [ ]:
BASE_PATH = "/content/drive/MyDrive/AI Resume Screening/"

DATA_PATH = BASE_PATH + "data/"
MODEL_PATH = BASE_PATH + "models/"
OUTPUT_PATH = BASE_PATH + "output/"

In [ ]:
resume_df = pd.read_csv(DATA_PATH + "processed_resume.csv")
job_df = pd.read_csv(DATA_PATH + "processed_jobs.csv")

resume_embeddings = np.load(MODEL_PATH + "resume_embeddings.npy")

index = faiss.read_index(MODEL_PATH + "job_faiss.index")

print(index.ntotal)

1615940


In [ ]:
with open(MODEL_PATH + "embedding_model.txt", "r") as f:
    MODEL_NAME = f.read().strip()

EMBEDDING_MODEL = SentenceTransformer(MODEL_NAME)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load Cross Encoder
CROSS_ENCODER_PATH = MODEL_PATH + "resume_matching_model/"

tokenizer = AutoTokenizer.from_pretrained(CROSS_ENCODER_PATH)

cross_encoder = AutoModelForSequenceClassification.from_pretrained(
    CROSS_ENCODER_PATH
)

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device
cross_encoder.to(device)

# Set evaluation mode
cross_encoder.eval()

print("✅ Cross Encoder Loaded Successfully!")
print(f"Device: {device}")

✅ Cross Encoder Loaded Successfully!
Device: cpu


In [ ]:
def extract_resume_text(file_path):

    text = ""

    if file_path.endswith(".pdf"):

        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"

    elif file_path.endswith(".docx"):

        doc = docx.Document(file_path)

        for para in doc.paragraphs:
            text += para.text + "\n"

    elif file_path.endswith(".txt"):
      with open(file_path,"r",encoding="utf-8") as f:
        text = f.read()

    else:
        raise ValueError("Only PDF and DOCX files are supported.")

    return text

In [ ]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", " ", text)

    text = re.sub(r"[^a-zA-Z0-9 ]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
def get_resume_embedding(resume_text):
  with torch.no_grad():
    embedding = EMBEDDING_MODEL.encode(
        resume_text,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

    return embedding.astype("float32")

In [ ]:
TOP_K = 50

def retrieve_jobs(resume_embedding):

    scores, indices = index.search(
        resume_embedding.reshape(1, -1),
        TOP_K
    )

    jobs = job_df.iloc[indices[0]].copy()

    jobs["faiss_score"] = scores[0]

    return jobs.reset_index(drop=True)

# Cross-Encoder Re-ranking

In [ ]:
import torch

device = next(cross_encoder.parameters()).device

def rerank_jobs(resume_text, retrieved_jobs):

    cross_encoder.eval()

    # Create (resume, job) pairs
    pairs = [(resume_text, row["job_text"]) for _, row in retrieved_jobs.iterrows()]

    # Tokenize all pairs together
    inputs = tokenizer(
        [pair[0] for pair in pairs],
        [pair[1] for pair in pairs],
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Run Cross Encoder once
    with torch.no_grad():
        outputs = cross_encoder(**inputs)
        scores = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()

    # Add scores and sort
    retrieved_jobs = retrieved_jobs.copy()
    retrieved_jobs["cross_encoder_score"] = scores
    retrieved_jobs = retrieved_jobs.sort_values(
        by="cross_encoder_score",
        ascending=False
    )

    return retrieved_jobs.reset_index(drop=True)

In [ ]:
resume_text = resume_df.iloc[0]["resume_text"]

resume_embedding = get_resume_embedding(resume_text)

retrieved_jobs = retrieve_jobs(resume_embedding)

ranked_jobs = rerank_jobs(
    resume_text,
    retrieved_jobs
)

ranked_jobs.head(10)

,Experience,Qualifications,Salary Range,location,Country,Work Type,Job Title,Role,Job Description,Benefits,skills,Responsibilities,Company,job_text,raw_job,faiss_score,cross_encoder_score
0,2 to 15 Years,MBA,$56K-$107K,Vientiane,Lao PDR,Temporary,Veterinarian,Equine Veterinarian,An Equine Veterinarian specializes in the heal...,"{'Casual Dress Code, Social and Recreational A...",Equine veterinary medicine Horse anatomy and p...,Specialize in the care and treatment of horses...,Agricultural Bank of China,veterinarian equine veterinarian an equine vet...,Veterinarian Equine Veterinarian An Equine Vet...,0.680604,0.507003
1,4 to 8 Years,BBA,$64K-$112K,Funafuti,Tuvalu,Temporary,Software Engineer,Backend Developer,A Backend Developer is tasked with building an...,"{'Employee Referral Programs, Financial Counse...",Proficiency in one or more backend programming...,Develop server-side logic and databases using ...,McDonald's,software engineer backend developer a backend ...,Software Engineer Backend Developer A Backend ...,0.680604,0.506913
2,4 to 8 Years,PhD,$57K-$87K,Accra,Ghana,Full-Time,Software Engineer,Frontend Developer,A Frontend Developer is responsible for design...,"{'Casual Dress Code, Social and Recreational A...","Proficiency in HTML, CSS, and JavaScript Exper...",Design and implement user interfaces for web a...,"Fidelity Investments, Inc.",software engineer frontend developer a fronten...,Software Engineer Frontend Developer A Fronten...,0.679628,0.506912
3,0 to 13 Years,PhD,$65K-$118K,Kabul,Afghanistan,Part-Time,Landscape Designer,Commercial Landscape Architect,Commercial Landscape Architects design outdoor...,"{'Casual Dress Code, Social and Recreational A...",Landscape architecture Site planning Plant sel...,Plan and design outdoor spaces for commercial ...,Franklin Resources,landscape designer commercial landscape archit...,Landscape Designer Commercial Landscape Archit...,0.679968,0.506864
4,2 to 9 Years,B.Tech,$55K-$84K,Gibraltar,Gibraltar,Temporary,Mechanical Designer,HVAC Systems Designer,"Design and optimize heating, ventilation, and ...","{'Legal Assistance, Bonuses and Incentive Prog...",HVAC system design Load calculations Energy ef...,"Specialize in designing heating, ventilation, ...",CarMax,mechanical designer hvac systems designer desi...,Mechanical Designer HVAC Systems Designer Desi...,0.679628,0.506862
5,2 to 12 Years,MBA,$63K-$115K,San Salvador,El Salvador,Contract,Dental Hygienist,Periodontal Therapist,A Periodontal Therapist specializes in treatin...,"{'Life and Disability Insurance, Stock Options...",Periodontal therapy procedures Patient care an...,Specialize in the treatment of gum diseases an...,Sainsbury's,dental hygienist periodontal therapist a perio...,Dental Hygienist Periodontal Therapist A Perio...,0.679628,0.506855
6,5 to 9 Years,BA,$60K-$110K,Zagreb,Croatia,Part-Time,Social Media Manager,Social Media Analyst,Social Media Analysts analyze social media dat...,"{'Casual Dress Code, Social and Recreational A...","Social media analytics tools (e.g., Google Ana...",Analyze social media data and user behavior to...,JSW Steel,social media manager social media analyst soci...,Social Media Manager Social Media Analyst Soci...,0.679628,0.506818
7,5 to 13 Years,M.Com,$60K-$104K,Cairo,Egypt,Full-Time,Landscape Architect,Urban Planner,Develop and implement urban planning strategie...,"{'Tuition Reimbursement, Stock Options or Equi...",Urban design and planning Zoning regulations G...,"Plan and design urban spaces, parks, and recre...",Markel,landscape architect urban planner develop and ...,Landscape Architect Urban Planner Develop and ...,0.680143,0.506817
8,5 to 12 Years,B.Tech,$55K-$120K,Montevideo,Uruguay,Intern,Data Engineer,ETL Developer,An ETL Developer specializes in designing and ...,"{'Life and Disability Insurance, Stock Options...","ETL (Extract, Transform, Load) processes Data ...","Extract, transform, and load (ETL) data from v...",Olin,data engineer etl developer an etl developer s.

In [ ]:
# print(resume_text[:3000])

hr administrator/marketing associate hr administrator summary dedicated customer service manager with 15+ years of experience in hospitality and customer service management. respected builder and leader of customer focused teams strives to instill a shared enthusiastic commitment to customer service. highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multi tasker client relations specialist accomplishments missouri dot supervisor training certification certified by ihg in customer loyalty and marketing by segment hilton worldwide general manager training certification accomplished trainer for cross server hospitality systems such as hilton onq micros opera pms fidelio opera reservation system ors holidex completed courses and seminars in customer service sales strategies inventory control loss prevention safety time management leadership and performance assessment. experience hr administrator/marke

In [ ]:
print(ranked_jobs[
    ["Job Title", "Company", "cross_encoder_score"]
].head(10))

              Job Title                     Company  cross_encoder_score
0          Veterinarian  Agricultural Bank of China             0.507003
1     Software Engineer                  McDonald's             0.506913
2     Software Engineer  Fidelity Investments, Inc.             0.506912
3    Landscape Designer          Franklin Resources             0.506864
4   Mechanical Designer                      CarMax             0.506862
5      Dental Hygienist                 Sainsbury's             0.506855
6  Social Media Manager                   JSW Steel             0.506818
7   Landscape Architect                      Markel             0.506817
8         Data Engineer                        Olin             0.506814
9    Aerospace Engineer              Continental AG             0.506810


In [ ]:
for i in [0, 100, 500, 1000]:
    print("=" * 100)

    resume_text = resume_df.iloc[i]["resume_text"]

    print("Resume:")
    print(resume_text[:200])

    resume_embedding = get_resume_embedding(resume_text)

    retrieved_jobs = retrieve_jobs(resume_embedding)

    ranked_jobs = rerank_jobs(resume_text, retrieved_jobs)

    print("\nTop 5 Jobs:")
    print(ranked_jobs[["Job Title", "Company", "cross_encoder_score"]].head())

Resume:
hr administrator/marketing associate hr administrator summary dedicated customer service manager with 15+ years of experience in hospitality and customer service management. respected builder and lead

Top 5 Jobs:
             Job Title                     Company  cross_encoder_score
0         Veterinarian  Agricultural Bank of China             0.507003
1    Software Engineer                  McDonald's             0.506913
2    Software Engineer  Fidelity Investments, Inc.             0.506912
3   Landscape Designer          Franklin Resources             0.506864
4  Mechanical Designer                      CarMax             0.506862
Resume:
hr benefits/leave coordinator summary 13 years of human resources experience and 27 years of administrative experience working in various settings professional detail oriented excellent time managemen

Top 5 Jobs:
                      Job Title                            Company  \
0             Software Engineer                       